<a href="https://colab.research.google.com/github/caio-fiap/CP05-SERS/blob/main/CheckPoint5_sers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Exercícios

treinar um modelo preditivo de classificação binária. O modelo fará as previsões com base nos dados da coluna `stabf` mostrando se a rede elétrica está INSTÁVEL ou ESTÁVEL. Utilizar o algoritmo Regressão Logística (Logistic Regression ou sklearn) para o treinamento do modelo. Avaliar o modelo com base na acurácia. Apresentar a Matriz de Confusão.


## Preparação do ambiente

In [55]:
# manipulacao de tabelas e matrizes/vetores
import pandas as pd
import numpy as np

# graficos
import matplotlib.pyplot as plt
import seaborn as sns

# algoritmo de regressao logistica
from sklearn.linear_model import LogisticRegression

# ferramenta para separacao dos dados de treino e teste
# X_train, X_test, y_train, y_test
from sklearn.model_selection import train_test_split

# avaliacao com acuracia e matriz de confusao
from sklearn.metrics import accuracy_score, confusion_matrix

## Carregar os dados e criar o dataframe

In [56]:
dados = pd.read_csv('https://raw.githubusercontent.com/caio-fiap/CP05-SERS/refs/heads/main/Data_for_UCI_named.csv')

In [57]:
dados.head()

,tau1,tau2,tau3,tau4,p1,p2,p3,p4,g1,g2,g3,g4,stab,stabf
0,2.959060,3.079885,8.381025,9.780754,3.763085,-0.782604,-1.257395,-1.723086,0.650456,0.859578,0.887445,0.958034,0.055347,unstable
1,9.304097,4.902524,3.047541,1.369357,5.067812,-1.940058,-1.872742,-1.255012,0.413441,0.862414,0.562139,0.781760,-0.005957,stable
2,8.971707,8.848428,3.046479,1.214518,3.405158,-1.207456,-1.277210,-0.920492,0.163041,0.766689,0.839444,0.109853,0.003471,unstable
3,0.716415,7.669600,4.486641,2.340563,3.963791,-1.027473,-1.938944,-0.997374,0.446209,0.976744,0.929381,0.362718,0.028871,unstable
4,3.134112,7.608772,4.943759,9.857573,3.525811,-1.125531,-1.845975,-0.554305,0.797110,0.455450,0.656947,0.820923,0.049860,unstable


## Realizar a inspeção básica dos dados

In [58]:
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   tau1    10000 non-null  float64
 1   tau2    10000 non-null  float64
 2   tau3    10000 non-null  float64
 3   tau4    10000 non-null  float64
 4   p1      10000 non-null  float64
 5   p2      10000 non-null  float64
 6   p3      10000 non-null  float64
 7   p4      10000 non-null  float64
 8   g1      10000 non-null  float64
 9   g2      10000 non-null  float64
 10  g3      10000 non-null  float64
 11  g4      10000 non-null  float64
 12  stab    10000 non-null  float64
 13  stabf   10000 non-null  object 
dtypes: float64(13), object(1)
memory usage: 1.1+ MB


In [59]:
# quantidade de (linhas, colunas)
dados.shape

(10000, 14)

In [60]:
# quais classes (valores) existem na coluna
dados['stabf'].unique()

array(['unstable', 'stable'], dtype=object)

In [61]:
# quantos registros existem de cada classe
dados['stabf'].value_counts()

,count
stabf,
unstable,6380
stable,3620


In [62]:
# estatisticas descritivas
# para atributos categoricos, usa se:
dados.describe(include=object)

,stabf
count,10000
unique,2
top,unstable
freq,6380


In [63]:
dados.columns

Index(['tau1', 'tau2', 'tau3', 'tau4', 'p1', 'p2', 'p3', 'p4', 'g1', 'g2',
       'g3', 'g4', 'stab', 'stabf'],
      dtype='object')

## Separação de dados de entrada (features) e dados de saida (target)

In [64]:
# features ---> caracteristicas que o modelo recebe para aprender os padroes no treino
# X (maiusculo) ---> variaveis independentes ---> FEATURES ---> DataFrame

X = dados.drop(['stab', 'stabf'], axis = 1) # exclui colunas e define o eixo (colunas)
X.head()

,tau1,tau2,tau3,tau4,p1,p2,p3,p4,g1,g2,g3,g4
0,2.959060,3.079885,8.381025,9.780754,3.763085,-0.782604,-1.257395,-1.723086,0.650456,0.859578,0.887445,0.958034
1,9.304097,4.902524,3.047541,1.369357,5.067812,-1.940058,-1.872742,-1.255012,0.413441,0.862414,0.562139,0.781760
2,8.971707,8.848428,3.046479,1.214518,3.405158,-1.207456,-1.277210,-0.920492,0.163041,0.766689,0.839444,0.109853
3,0.716415,7.669600,4.486641,2.340563,3.963791,-1.027473,-1.938944,-0.997374,0.446209,0.976744,0.929381,0.362718
4,3.134112,7.608772,4.943759,9.857573,3.525811,-1.125531,-1.845975,-0.554305,0.797110,0.455450,0.656947,0.820923


In [67]:
dados['stabf'] = dados['stabf'].replace({'unstable': 'instável', 'stable': 'estável'})
dados['stabf'].value_counts()

,count
stabf,
instável,6380
estável,3620


In [66]:
# target ---> atributo que sera previsto pelo modelo
# y (minusculo) ---> variavel dependente ---> TARGET ---> array numpy com dados da coluna (no exemplo, 'stabf')

y = dados['stabf']
y

,stabf
0,instável
1,estável
2,instável
3,instável
4,instável
...,...
9995,instável
9996,estável
9997,estável
9998,instável


## Separação de dados de treino e teste

In [70]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [72]:
# total de registros ----> 10,000
X_train.shape[0]

8000

In [73]:
X_test.shape[0]

2000

## Treinamento, geração de previsões e avaliação do modelo

In [76]:
# instanciando o modelo
modelo = LogisticRegression()

In [78]:
# treinamento
modelo.fit(X_train, y_train)

LogisticRegression()

In [81]:
# gerar o array de probabilidades
probabilidades = modelo.predict_proba(X_test)
probabilidades[:20]

array([[0.04609348, 0.95390652],
       [0.06082634, 0.93917366],
       [0.31665189, 0.68334811],
       [0.97845447, 0.02154553],
       [0.64488737, 0.35511263],
       [0.57310264, 0.42689736],
       [0.7293308 , 0.2706692 ],
       [0.0220369 , 0.9779631 ],
       [0.04455238, 0.95544762],
       [0.74263825, 0.25736175],
       [0.05821672, 0.94178328],
       [0.43539611, 0.56460389],
       [0.54821046, 0.45178954],
       [0.22628137, 0.77371863],
       [0.01738045, 0.98261955],
       [0.24059749, 0.75940251],
       [0.43383254, 0.56616746],
       [0.19235057, 0.80764943],
       [0.42956875, 0.57043125],
       [0.30915071, 0.69084929]])

In [83]:
# gerar previsoes
y_predict = modelo.predict(X_test)
y_predict[:20]

array(['instável', 'instável', 'instável', 'estável', 'estável',
       'estável', 'estável', 'instável', 'instável', 'estável',
       'instável', 'instável', 'estável', 'instável', 'instável',
       'instável', 'instável', 'instável', 'instável', 'instável'],
      dtype=object)

In [86]:
# avaliar com base na acuracia
acuracia = 100 * accuracy_score(y_test, y_predict)
print(f"A acurácia do modelo é de {acuracia:.2f}%")

A acurácia do modelo é de 81.70%


In [ ]:
# matriz de confusao com matplotlib e heatmap
